# **INFERENCIA CON MODELOS ENTRENADOS Y PREVIAMENTE GUARDADOS**

Objetivos:

* Cargar automáticamente los top modelos guardados (.joblib) y sus artefactos auxiliares.

* Reconstruir features de test exactamente como en entrenamiento (sin leakage):

 * Tabulares: temporal, espacial, institucional, lag (alineación de columnas).

 * Texto: matriz TF-IDF .npz (ya vectorizada).

 * Bloques: X_test_full_unscaled, X_test_full_scaled, y X_test_mlp (si lo usaste).

* Definir 3 casos de prueba, ejecutar predicción en todos los modelos cargados y reportar Top-3 clases por modelo.

| Archivo                               | Ubicación típica            | Obligatorio | ¿Para qué sirve?                                                                                                                        |
| ------------------------------------- | --------------------------- | ----------: | --------------------------------------------------------------------------------------------------------------------------------------- |
| `results/final_models/*.joblib`       | `RESULTS_DIR/final_models/` |          ✔️ | Modelos finales ya entrenados (p. ej., `U_XGBoost_hist,_n=500,_depth=6,_eta=0.08.joblib`, `M_ROS+MLP_128x64,_early_stop.joblib`, etc.). |
| `results/scaler_tabular.joblib`       | `RESULTS_DIR/`              |         ✔️* | `StandardScaler` ajustado **en train** para escalar las **columnas continuas** (solo para los modelos “Scaled”).                        |
| `results/scaler_continuous_cols.json` | `RESULTS_DIR/`              |         ✔️* | Lista de **columnas continuas** usadas al ajustar el `StandardScaler`.                                                                  |
| `results/label_encoder_y.joblib`      | `RESULTS_DIR/`              |        ✔️** | `LabelEncoder` entrenado con las clases del target (solo si algún modelo —como XGBoost— fue entrenado con etiquetas codificadas).       |
| `data/test_feat_temporal.csv`         | `DATA_DIR/`                 |          ✔️ | Bloque de features temporales de **test**.                                                                                              |
| `data/test_feat_spatial.csv`          | `DATA_DIR/`                 |          ✔️ | Bloque de features espaciales de **test**.                                                                                              |
| `data/test_feat_institutional.csv`    | `DATA_DIR/`                 |          ✔️ | Bloque de features institucionales de **test**.                                                                                         |
| `data/test_feat_lag.csv`              | `DATA_DIR/`                 |          ✔️ | Bloque de **lag/coherencia temporal** de **test**.                                                                                      |
| `data/tfidf_delito_test.npz`          | `DATA_DIR/`                 |          ✔️ | Matriz **TF-IDF** de **test** (sparse).                                                                                                 |
| `data/tfidf_delito_train.npz`         | `DATA_DIR/`                 |           ➖ | (No se usa en inferencia, pero útil para comprobaciones).                                                                               |
| `data/tfidf_delito_features.csv`      | `DATA_DIR/`                 |           ➖ | Vocabulario TF-IDF (informativo).                                                                                                       |
| `data/tfidf_vectorizer.joblib`        | `DATA_DIR/`                 |           ➖ | `TfidfVectorizer` original (solo si vas a transformar **texto crudo nuevo**).                                                           |
| `data/test_split.csv`                 | `DATA_DIR/`                 |          ✔️ | Metadatos del test, incluye la **etiqueta real** `categoria_delito` (para comparar).                                                    |



* Obligatorio para usar modelos que requieren escala (LogReg, SVC, MLP, KNN, etc.).
** Obligatorio para modelos como XGBoost u otros, con vector y codificado a enteros.

# **1. Librerias**

In [1]:
import os
import numpy as np
import pandas as pd
import joblib, json
from scipy import sparse
from scipy.sparse import hstack, csr_matrix
from pprint import pprint

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.extmath import softmax
from sklearn.metrics import top_k_accuracy_score

from sklearn.utils.extmath import softmax

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="The default value of `dual` will change")



# **2. CONFIGURACIÓN DEL AMBIENTE**


## 2.1 VARIABLES GLOBALES

In [2]:
# Semilla global para reproducibilidad
SEED = 42

# Le indicamos a Pandas que siempre muestre hasta 50 columnas completas en lugar de cortarlas con “...”.
pd.set_option("display.max_columns", 50)

## 2.2 ACCESO A MODELOS GUARDADOS

In [3]:
# ---- Switch de entorno: "colab" o "local"
RUN_ENV = "local"   # <-- cambia a "local" si no usas Google Colab

# ---- Rutas base
if RUN_ENV.lower() == "colab":
    try:
        from google.colab import drive
        drive.mount("/drive")
    except Exception as e:
        print("Aviso: no se pudo montar Drive automáticamente:", e)
    # Ajusta esta ruta a tu carpeta del proyecto en Drive
    BASE_DIR = "/drive/My Drive/Colab Notebooks/reto-Thales/"
else:
    # Ajusta a tu ruta local del proyecto
    BASE_DIR = "/Users/jorgevalverde/Documents/myEnv/crimen-cdmx/"

os.makedirs(BASE_DIR, exist_ok=True)

# Ruta de los modelos
DATA_DIR = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(RESULTS_DIR, "final_models")

print(f"Entorno: {RUN_ENV} | BASE_DIR: {BASE_DIR}")

Entorno: local | BASE_DIR: /Users/jorgevalverde/Documents/myEnv/crimen-cdmx/


In [4]:
# Vamos utilizar un archivo de funciones propio
#sys.path.insert(0, BASE_DIR) #para acceder desde Colab-Drive
import myUtils as myUtils


# **3. CARGA DE MODELOS**


In [5]:
# Cargar todos los modelos .joblib del directorio
model_files = [f for f in os.listdir(MODELS_DIR) if f.endswith(".joblib")]
models_loaded = {}

for f in model_files:
    name = os.path.splitext(f)[0]
    path = os.path.join(MODELS_DIR, f)
    try:
        models_loaded[name] = joblib.load(path)
        print(f" Cargado: {name}")
    except Exception as e:
        print(f"  No se pudo cargar {name}: {e}")

print(f"\nTotal modelos cargados: {len(models_loaded)}")

 Cargado: S_LinearSVC_C=0.4,_balanced
 Cargado: U_DecisionTree_depth=None,_min_leaf=1
 Cargado: U_XGBoost_hist,_n=500,_depth=6,_eta=0.08
 Cargado: U_RandomForest_n=500,_balanced_subsample
 Cargado: U_ExtraTrees_n=500,_depth=None
 Cargado: M_ROS+MLP_128x64,_early_stop

Total modelos cargados: 6


In [6]:
LE_PATH = os.path.join(RESULTS_DIR, "label_encoder_y.joblib")
label_encoder = None
if os.path.exists(LE_PATH):
    try:
        label_encoder = joblib.load(LE_PATH)
        print(" LabelEncoder cargado.")
    except Exception as e:
        print(f"  No se pudo cargar LabelEncoder: {e}")
else:
    print("  No se encontró label_encoder_y.joblib (no es obligatorio para este smoke test).")


 LabelEncoder cargado.


# **4. INFERENCIA/PREDICCIÓN**

## 4.1 Carga del modelo

In [7]:
# Ejecutar smoke test
_ = myUtils.smoke_test_models(models_loaded, label_encoder=label_encoder, topk=3)


S_LinearSVC_C=0.4,_balanced
  n_features_in_=1161
  Predicción: delito de bajo impacto

U_DecisionTree_depth=None,_min_leaf=1
  n_features_in_=1161
  Top-3:
   1. delito de bajo impacto  (p=1.000)
   2. robo de vehiculo con y sin violencia  (p=0.000)
   3. robo a transeunte en via publica con y sin violencia  (p=0.000)

U_XGBoost_hist,_n=500,_depth=6,_eta=0.08
  n_features_in_=1161
  Top-3:
   1. delito de bajo impacto  (p=1.000)
   2. hecho no delictivo  (p=0.000)
   3. otras  (p=0.000)

U_RandomForest_n=500,_balanced_subsample
  n_features_in_=1161
  Top-3:
   1. delito de bajo impacto  (p=0.480)
   2. otras  (p=0.406)
   3. hecho no delictivo  (p=0.112)

U_ExtraTrees_n=500,_depth=None
  n_features_in_=1161
  Top-3:
   1. otras  (p=0.508)
   2. delito de bajo impacto  (p=0.336)
   3. hecho no delictivo  (p=0.156)

M_ROS+MLP_128x64,_early_stop
  n_features_in_=448
  Top-3:
   1. delito de bajo impacto  (p=0.967)
   2. otras  (p=0.030)
   3. hecho no delictivo  (p=0.001)


Con esto, validamos que los modelos se cargan y admiten un vector con el número de columnas correcto.

In [8]:
# USaremos el modelo XGBoost

#  Localizar el modelo XGBoost cargado
xgb_key = None
for k in models_loaded.keys():
    if "XGBoost" in k or "XGB" in k:
        xgb_key = k
        break

if xgb_key is None:
    raise RuntimeError("No se encontró un modelo XGBoost en 'models_loaded'.")

xgb_model = models_loaded[xgb_key]
nfeat = getattr(xgb_model, "n_features_in_", None)
if nfeat is None:
    raise RuntimeError(f"El modelo {xgb_key} no expone n_features_in_.")

print(f"Usando modelo: {xgb_key} | n_features_in_ = {nfeat}")

Usando modelo: U_XGBoost_hist,_n=500,_depth=6,_eta=0.08 | n_features_in_ = 1161


## 4.2 Testar en una instancia dummy

In [9]:
# Construir una instancia dummy con 'señales' (valores > 0 en algunos índices)
#    - Elegimos varios índices "seguros" (< nfeat)
rng = np.random.default_rng(42)

# Escoge unos 10 índices bien repartidos dentro del rango [0, nfeat)
k_active = min(10, max(5, nfeat // 200))  # 5 a 10 señales aprox.
idx_candidates = np.linspace(0, nfeat-1, num=50, dtype=int)  # 50 puntos uniformes
active_idx = rng.choice(idx_candidates, size=k_active, replace=False)
active_idx.sort()

# Asignamos valores "tipo TF-IDF/tabular" moderados
active_vals = rng.uniform(0.3, 1.0, size=k_active).astype(np.float32)

x_dummy = csr_matrix(
    (active_vals, (np.zeros_like(active_idx), active_idx)),
    shape=(1, nfeat),
    dtype=np.float32
)

# Algunas señales activas, el resto en cero
print(f"Dummy con {k_active} señales activas en índices: {active_idx.tolist()}")

Dummy con 5 señales activas en índices: [94, 497, 733, 852, 1160]


In [10]:
# Predicción con probabilidades (multi-clase)
if hasattr(xgb_model, "predict_proba"):
    probs = xgb_model.predict_proba(x_dummy)[0]
    classes = getattr(xgb_model, "classes_", None)
    # Decodificar si tenemos LabelEncoder y las clases del modelo son enteras
    if (label_encoder is not None and classes is not None 
        and np.issubdtype(np.asarray(classes).dtype, np.integer)):
        class_labels = label_encoder.inverse_transform(classes)
    else:
        class_labels = classes if classes is not None else np.arange(len(probs))

    # Top-k
    topk = 5 if len(probs) >= 5 else len(probs)
    order = np.argsort(probs)[::-1][:topk]
    print("\nTop-{} (clase, prob):".format(topk))
    for r, j in enumerate(order, 1):
        print(f" {r}. {class_labels[j]}  (p={probs[j]:.3f})")

else:
    # fallback si no hay predict_proba (no debería pasar con XGBClassifier)
    pred = xgb_model.predict(x_dummy)[0]
    # decodificar si es entero y hay LabelEncoder
    if label_encoder is not None and isinstance(pred, (int, np.integer)):
        pred = label_encoder.inverse_transform([pred])[0]
    print(f"Predicción: {pred}")


Top-5 (clase, prob):
 1. delito de bajo impacto  (p=1.000)
 2. hecho no delictivo  (p=0.000)
 3. otras  (p=0.000)
 4. robo a transeunte en via publica con y sin violencia  (p=0.000)
 5. robo de vehiculo con y sin violencia  (p=0.000)


## 4.3 Testar en instancia real

In [11]:
artifacts = {}

paths = {
        "xgb": os.path.join(BASE_DIR, "results/final_models/U_XGBoost_hist,_n=500,_depth=6,_eta=0.08.joblib"),
        "tfidf": os.path.join(BASE_DIR, "data/tfidf_vectorizer.joblib"),
        "encoder": os.path.join(BASE_DIR, "results/label_encoder_y.joblib"),
        "tabcols": os.path.join(BASE_DIR, "results/tabular_columns.json"),
}

for key, path in paths.items():
    if os.path.exists(path):
        artifacts[key] = joblib.load(path) if path.endswith(".joblib") else json.load(open(path))
    else:
        print(f" Advertencia: artefacto {key} no encontrado en {path}")
        artifacts[key] = None

In [12]:
pprint({k: type(v) for k, v in artifacts.items()})

{'encoder': <class 'sklearn.preprocessing._label.LabelEncoder'>,
 'tabcols': <class 'list'>,
 'tfidf': <class 'sklearn.feature_extraction.text.TfidfVectorizer'>,
 'xgb': <class 'xgboost.sklearn.XGBClassifier'>}


In [13]:
# Simulación de entrada humana
df_humano = pd.DataFrame([{
    "delito": "robo de vehículo con violencia en vía pública",
    "fecha_inicio": "2025-10-10",
    "hora_inicio": "22:30",
    "fecha_hecho": "2025-10-10",
    "hora_hecho": "22:00",
    "latitud": 19.43,
    "longitud": -99.14,
    "anio_inicio": 2025,
    "mes_inicio": "octubre",
    "alcaldia_hecho": "CUAUHTÉMOC",
    "colonia_hecho": "CENTRO",
    "competencia": "LOCAL",
    "fiscalia": "FGJCDMX",
    "agencia": "AGENCIA INVESTIGADORA SUR",
    "unidad_investigacion": "UI ROBO DE VEHÍCULOS"
}])

df_humano

,delito,fecha_inicio,hora_inicio,fecha_hecho,hora_hecho,latitud,longitud,anio_inicio,mes_inicio,alcaldia_hecho,colonia_hecho,competencia,fiscalia,agencia,unidad_investigacion
0,robo de vehículo con violencia en vía pública,2025-10-10,22:30,2025-10-10,22:00,19.43,-99.14,2025,octubre,CUAUHTÉMOC,CENTRO,LOCAL,FGJCDMX,AGENCIA INVESTIGADORA SUR,UI ROBO DE VEHÍCULOS


In [14]:
results = myUtils.predict_xgb_from_raw(df_humano, artifacts, topk=3)
print("\nPredicción para caso humano:")
for r in results:
    print(f" Caso {r['caso']}:")
    for i, (cls, prob) in enumerate(r["topk"], 1):
        print(f"  {i}. {cls:50s} (p={prob:.3f})")


Feature vector de la instancia a predecir:
  (np.int32(0), np.int32(0))	2025.0
  (np.int32(0), np.int32(1))	10.0
  (np.int32(0), np.int32(2))	4.0
  (np.int32(0), np.int32(5))	-0.8660253882408142
  (np.int32(0), np.int32(6))	0.5
  (np.int32(0), np.int32(7))	-0.3826834261417389
  (np.int32(0), np.int32(8))	0.9238795042037964
  (np.int32(0), np.int32(9))	19.43000030517578
  (np.int32(0), np.int32(10))	-99.13999938964844
  (np.int32(0), np.int32(1157))	0.18432220816612244
  (np.int32(0), np.int32(1120))	0.36924466490745544
  (np.int32(0), np.int32(1119))	0.36924466490745544
  (np.int32(0), np.int32(1115))	0.5219531059265137
  (np.int32(0), np.int32(1100))	0.3270430266857147
  (np.int32(0), np.int32(928))	0.39134567975997925
  (np.int32(0), np.int32(859))	0.17811885476112366
  (np.int32(0), np.int32(802))	0.35926610231399536
Shape final: (1, 1161)

Predicción para caso humano:
 Caso 0:
  1. delito de bajo impacto                             (p=0.984)
  2. robo de vehiculo con y sin violenci